# Notebook 1: Register Campaign + Station, Upload Example Data

This notebook is intentionally simple and editable.

Flow:

1. Connect to Upstream
2. Create (or reuse) a campaign
3. Create (or reuse) a station
4. Upload `data/sensors.csv` and `data/measurements.csv`
5. Save IDs for Notebook 2

## Imports

In [1]:
import json
from datetime import datetime, timedelta, timezone
from getpass import getpass
from pathlib import Path

from upstream.client import UpstreamClient
from upstream_api_client.models import CampaignsIn, StationCreate

from utils import safe_items, safe_total

## User Inputs

Edit this cell every run as needed.

In [2]:
UPSTREAM_USERNAME = input("UPSTREAM_USERNAME: " ).strip()
UPSTREAM_PASSWORD = getpass("UPSTREAM_PASSWORD: ")
UPSTREAM_BASE_URL =  'https://dsoinstitute2026api.pods.portals.tapis.io/'

CREATE_NEW_CAMPAIGN = True
CAMPAIGN_ID = 1  # set this when CREATE_NEW_CAMPAIGN = False

CREATE_NEW_STATION = True
STATION_ID = None  # set this when CREATE_NEW_STATION = False

CONTACT_NAME = "SDK Tutorial"
CONTACT_EMAIL = "tutorial@example.com"

## Paths

In [3]:
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "data"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
FLOW_CONTEXT_PATH = OUTPUT_DIR / "flow_context.json"

SENSORS_FILE = DATA_DIR / "sensors.csv"
MEASUREMENTS_FILE = DATA_DIR / "measurements.csv"

print("Sensors file:", SENSORS_FILE)
print("Measurements file:", MEASUREMENTS_FILE)

Sensors file: /Users/wmobley/Documents/GitHub/upstream/upstream-sdk/tutorials/basic-sdk-flow/data/sensors.csv
Measurements file: /Users/wmobley/Documents/GitHub/upstream/upstream-sdk/tutorials/basic-sdk-flow/data/measurements.csv


## Connect

In [4]:
client = UpstreamClient(
    username=UPSTREAM_USERNAME,
    password=UPSTREAM_PASSWORD,
    base_url=UPSTREAM_BASE_URL,
)

print("Authenticated:", client.authenticate())

Authenticated: True


## Create or Reuse Campaign

In [5]:
run_label = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")

if CREATE_NEW_CAMPAIGN:
    campaign_payload = CampaignsIn(
        name=f"Upstream Tutorial Campaign {run_label}",
        description="Campaign created by tutorial notebook 1",
        contact_name=CONTACT_NAME,
        contact_email=CONTACT_EMAIL,
        allocation='dso-institute',
        start_date=datetime.now(timezone.utc),
        end_date=datetime.now(timezone.utc) + timedelta(days=30),
    )
    created_campaign = client.create_campaign(campaign_payload)
    CAMPAIGN_ID = created_campaign.id

print("CAMPAIGN_ID:", CAMPAIGN_ID)

CAMPAIGN_ID: 3


## Create or Reuse Station

In [6]:
if CREATE_NEW_STATION:
    station_payload = StationCreate(
        name=f"SDK Tutorial Station {run_label}",
        description="Station created by tutorial notebook 1",
        contact_name=CONTACT_NAME,
        contact_email=CONTACT_EMAIL,
        start_date=datetime.now(timezone.utc),
        active=True,
    )
    created_station = client.create_station(campaign_id=CAMPAIGN_ID, station_create=station_payload)
    STATION_ID = created_station.id

print("STATION_ID:", STATION_ID)

STATION_ID: 3


## Validate and Upload Data

In [7]:
validation = client.validate_files(
    sensors_file=SENSORS_FILE,
    measurements_file=MEASUREMENTS_FILE,
)
print(validation)

upload_result = client.upload_csv_data(
    campaign_id=CAMPAIGN_ID,
    station_id=STATION_ID,
    sensors_file=SENSORS_FILE,
    measurements_file=MEASUREMENTS_FILE,
)
print(upload_result)

{'valid': True, 'sensors_validation': {'valid': True, 'sensor_count': 3, 'message': 'Validated 3 sensors'}, 'measurements_validation': {'valid': True, 'measurement_count': 8961, 'message': 'Validated 8961 measurements'}, 'message': 'All files validated successfully'}
{'success': True, 'campaign_id': 3, 'station_id': 3, 'sensors_file': '/Users/wmobley/Documents/GitHub/upstream/upstream-sdk/tutorials/basic-sdk-flow/data/sensors.csv', 'measurements_file': '/Users/wmobley/Documents/GitHub/upstream/upstream-sdk/tutorials/basic-sdk-flow/data/measurements.csv', 'response': {'uploaded_file_sensors stored in memory': True, 'uploaded_file_measurements stored in memory': True, 'Total sensors processed': 3, 'Total measurements added to database': -3, 'Data Processing time': '19.6 seconds.', 'errors': []}, 'message': 'Data uploaded successfully'}


## Quick Check

In [8]:
sensors_page = client.sensors.list(campaign_id=CAMPAIGN_ID, station_id=STATION_ID, limit=100, page=1)
sensors = safe_items(sensors_page)
print("Sensor total:", safe_total(sensors_page))

for sensor in sensors:
    print(sensor.id, getattr(sensor, "alias", None), getattr(sensor, "units", None))

if sensors:
    first_sensor_id = sensors[0].id
    measurements_page = client.list_measurements(
        campaign_id=CAMPAIGN_ID,
        station_id=STATION_ID,
        sensor_id=first_sensor_id,
        limit=10,
        page=1,
    )
    print("Measurement total for first sensor:", safe_total(measurements_page))

Sensor total: 3
1 Rain Increment inches
2 Flow Volume cfs
3 River Stage ft
Measurement total for first sensor: 8959


## Save Context for Notebook 2

In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
flow_context = {
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "base_url": UPSTREAM_BASE_URL,
    "campaign_id": CAMPAIGN_ID,
    "station_id": STATION_ID
}
FLOW_CONTEXT_PATH.write_text(json.dumps(flow_context, indent=2), encoding="utf-8")
print(FLOW_CONTEXT_PATH)
print(flow_context)

/Users/wmobley/Documents/GitHub/upstream/upstream-sdk/tutorials/basic-sdk-flow/outputs/flow_context.json
{'saved_at_utc': '2026-05-10T19:43:47.503974+00:00', 'base_url': 'https://dsoinstitute2026api.pods.portals.tapis.io/', 'campaign_id': 3, 'station_id': 3}
